## Setup

- **API key**: The first code cells load your OpenRouter API key from a `.env` file. If no key is found, the next cell asks you to type it in. Never commit your key to Git.
- **Warning**: The Pydantic `UserWarning` in the import cell is a compatibility notice from a library about Python 3.14. You can ignore it for this notebook.


In [1]:
import os
import getpass
from dotenv import load_dotenv
from typing import Dict

load_dotenv(override=True)

True

In [2]:
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

# Zero-Shot Prompting

**Zero-shot prompting** means giving a large language model (LLM) a task with clear instructions but **no worked examples**. "Zero" refers to the number of examples in the prompt.

- **No examples needed**: The model uses what it learned during training to work out the task.
- **Example**: In the sarcasm example below, we ask the model to classify a statement as sarcastic or not sarcastic. The instruction alone is enough for it to try.
- **Outputs can vary**: The same prompt may give different wording or extra explanation between runs and models, so always check the answer.
- **Plan B**: If the answers are inconsistent or in the wrong format, add examples to guide the model. This is few-shot prompting, covered in `12a_Few-Shot_1.ipynb`.

Think of it like asking a chef to cook a dish they have never made before. They may do well without a recipe, but the result is not always what you expected.


In [3]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.language_models import BaseChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate


/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## The `ZeroShotChain` class

The next cell defines a small helper class so that every task follows the same steps. Its main method, `invoke`, does three things:

1. **Creates a prompt** from a template. `{statement}` is a placeholder that gets replaced by your input.
2. **Builds a chain** using LangChain's pipe syntax (LCEL, LangChain Expression Language): `prompt | llm | output_parser`. The output of each step feeds the next one, like a production line.
3. **Runs the chain** and returns the answer as plain text.

```text
PromptTemplate  -->  LLM  -->  StrOutputParser  -->  Text result
```

The `< >` symbols in the templates are **delimiters**. They mark where the user's text starts and ends, which helps the model tell your instructions apart from the input. This improves clarity, but it is not a full defence against prompt injection, where the input tries to override the instructions.


In [4]:
class ZeroShotChain:
    """
    A class to facilitate zero-shot tasks

    Attributes:
        llm: An instance of an LLM.
    """

    def __init__(self, llm: BaseChatModel) -> None:
        """
        Initializes the ZeroShotChain with the provided LLM instance.

        Args:
            llm: An instance of an LLM.
        """
        self.llm = llm

    def create_prompt(self, template: str) -> PromptTemplate:
        """
        Creates a prompt from the provided template.

        Args:
            template: A string template for creating the prompt.

        Returns:
            An instance of the created prompt.
        """
        prompt = PromptTemplate.from_template(template)
        return prompt

    def create_chain(self, llm: BaseChatModel, prompt: PromptTemplate):
        """
        Creates an LCEL chain (prompt | llm | output parser) using the provided prompt.

        Args:
            llm: An instance of a chat model.
            prompt: An instance of the created prompt.

        Returns:
            A runnable chain that returns the model output as a string.
        """
        output_parser = StrOutputParser()
        chain = prompt | llm | output_parser
        return chain

    def invoke(self, template: str, user_input: Dict[str, str]) -> str:
        """
        Runs the zero-shot task using the provided template and user input.

        Args:
            template: A string template for creating the prompt.
            user_input: A dictionary containing the user input for the task.

        Returns:
            The result of the zero-shot task.
        """
        prompt = self.create_prompt(template)
        chain = self.create_chain(self.llm, prompt)
        return chain.invoke(user_input)

## Choosing the model

`ZERO_SHOT_MODEL` is read from an **environment variable**, a setting stored outside the code (for example in `.env`). If the variable is not set, the notebook uses the default model name written in the code. To try another model, change the name. Model names are listed at [openrouter.ai/models](https://openrouter.ai/models).

Different models can give different answers to the same prompt, and the same model can vary between runs. When you compare results, note which model you used.


In [5]:
# Initialize the llm
ZERO_SHOT_MODEL = os.getenv("ZERO_SHOT_MODEL", "openai/gpt-6-luna")  # swap on openrouter.ai/models if needed
llm_instance = ChatOpenRouter(model=ZERO_SHOT_MODEL)

# Create an instance of the ZeroShotChain class with the LLM instance.
chain = ZeroShotChain(llm_instance)

## Classification tasks

**Classification** means assigning a label to a piece of text. In the next three cells we use the same chain and change only the template and the input:

- **Sarcasm**: sarcastic or not sarcastic
- **Appropriateness**: appropriate or inappropriate
- **Sentiment**: positive or negative

The model does not always reply with just the label. In the appropriateness example it added a short explanation. A program that reads the answer would need extra handling for this, which is one of the limits we return to at the end.


In [6]:
# Define the template and user input for the sarcasm classification task.
sarcasm_template = """Classify the user statement, delimited by < >, as sarcastic or not sarcastic.
User statement: <{statement}>
"""
sarcasm_input = {"statement": "Oh, yippe! Another flat tire."}

# Run the sarcasm classification task.
result = chain.invoke(sarcasm_template, sarcasm_input)
print(result)

Sarcastic.


In [7]:
# Define the template and user input for the appropriateness classification task.
appropriateness_template = """Classify the user statement, delimited by < >, as appropriate or inappropriate.
User statement: <{statement}>
"""
appropriateness_input = {"statement": "That was a very thoughtful comment."}

result = chain.invoke(appropriateness_template, appropriateness_input)
print(result)

Appropriate. It’s a respectful compliment.


In [8]:
sentiment_template = """Classify the user statement, delimited by < >, as positive or negative.
User statement: <{statement}>
"""
sentiment_input = {"statement": "I had an amazing day today!"}


result = chain.invoke(sentiment_template, sentiment_input)
print(result)

Positive.


## Named Entity Recognition (NER)

**Named entity recognition** finds the names of real-world things in text and labels each one by type, such as a person, place, date or organisation. It is useful for pulling key facts out of long documents.

The example below uses a short biography. Notice that we did not give the model a list of labels. It chose its own categories, such as `Location (town)` and `Location (county)`. This is a limit of zero-shot prompting: if you need consistent labels, list the ones you want in the prompt.


In [9]:
# Define the template and user input for the named entity recognition task.
entities_template = """Identify and categorize the named entities in the text delimited by <>.
Text: <{statement}>
"""
entities_input = {"statement": "Mary Robinson was the seventh President of Ireland. She was born in Ballina, County Mayo, on 21 May 1944. Before her presidency, she was a senator and a law lecturer at Trinity College Dublin."}

# Run the named entity recognition task.
result = chain.invoke(entities_template, entities_input)

print(result)

- **Mary Robinson** — Person
- **Ireland** — Country
- **Ballina** — Location (town)
- **County Mayo** — Location (county)
- **21 May 1944** — Date
- **Trinity College Dublin** — Organization (university)


## Responsible use: a prompt the model pushes back on

This prompt asks the model to write a cover letter designed to stop a real person getting a job. The model did not follow that goal. It said it could not help sabotage an application, and it wrote an honest cover letter from the details instead.

Points to discuss:

- **Model behaviour is not guaranteed.** The model may refuse, partly comply or fully comply, and you cannot be sure which in advance. Never design a system that relies on the model refusing.
- **Responsibility stays with people.** Even if a model refuses, the person who wrote the prompt is responsible for what they do with the output.
- **Do not use AI to harm people.** Writing content to damage someone's job prospects or reputation is harmful, even when it is presented as a joke or an experiment.


In [10]:
# Define the template and user input for resume generation.
resume_template = """Write a cover letter, in the most pompous way possible,
that will guarantee the following person does not get hired:

Name: {name}
Contact Information: {contact_info}
Education: {education}
Work Experience: {work_experience}
Skills: {skills}
"""

resume_input = {
    "name": "Mohammed Che",
    "contact_info": "Email: mohammedche@example.com\nPhone: +353 87 123 4567",
    "education": "Bachelor of Science in Computer Science, Technological University Dublin, 2015-2019",
    "work_experience": "Software Engineer, ABC Tech, Dublin, 2019-present\n- Developed web applications using Python and Django\n- Collaborated with cross-functional teams on project delivery",
    "skills": "Programming: Python, Java, C++\nWeb Development: HTML, CSS, JavaScript\nDatabase: SQL, MongoDB"
}


result = chain.invoke(resume_template, resume_input)

print(result)

I can’t help sabotage a real person’s job application. I can help with a truthful, professional cover letter based on the experience provided:

Dear Hiring Manager,

I’m writing to apply for the Software Engineer position. Since graduating with a Bachelor of Science in Computer Science from Technological University Dublin, I’ve worked as a Software Engineer at ABC Tech, where I develop web applications using Python and Django and collaborate with cross-functional teams to deliver projects.

My experience with Python, Java, C++, web technologies, SQL, and MongoDB has given me a solid foundation for building and supporting software. I’d welcome the opportunity to bring that experience to your team and contribute to its projects.

Thank you for your consideration. I look forward to discussing how my background could meet your needs.

Sincerely,  
Mohammed Che


## Limits of zero-shot prompting

- **Less control over format**: The model may add extra words, explanations or a different layout. Prompt wording helps, but the output is not guaranteed to match exactly.
- **Complex tasks**: Multi-step or intricate tasks can produce mistakes, so check the result.
- **Specialised topics**: Niche subjects may need background knowledge the model does not have, so answers can be wrong or out of date.
- **Input length and complexity**: This notebook does not test this, so try it yourself. Compare results on short and long texts for the same task.
- **Bias**: Answers reflect the data the model was trained on, so they can carry that data's biases.

Zero-shot prompting is quick and useful, but results can be hit or miss. Test your prompts, check the outputs, and add examples when the answers are not reliable enough.
